In [1]:
from datasets import load_dataset
from collections import Counter
import gc
import ast
import pandas as pd

In [2]:
cleaned_df = pd.read_parquet("danbooru2025_cleaned.parquet")

In [16]:
# ---------------------------------
# Step 1: Filter
# ---------------------------------

lite_df = cleaned_df[
    (cleaned_df["score"] > 50) &
    (cleaned_df["rating"] == "g")
].copy()
lite_df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url
353,9155913,2025-04-15T13:22:23.565-04:00,54,g,15591835,1girl black_sclera blonde_hair broken_mask col...,trigger_(zenless_zone_zero),zenless_zone_zero,https://cdn.donmai.us/360x360/33/9f/339f49cfdb...
1495,9151749,2025-04-14T20:17:09.437-04:00,60,g,402201,1girl alternate_costume bare_shoulders black_h...,ellen_joe,zenless_zone_zero,https://cdn.donmai.us/360x360/63/86/6386a7c990...
3069,9146829,2025-04-13T12:31:41.464-04:00,54,g,371585,1girl black_hair black_pantyhose blue_leotard ...,mona_(genshin_impact),genshin_impact,https://cdn.donmai.us/360x360/17/4c/174ca60b01...
3106,9146730,2025-04-13T12:10:46.934-04:00,53,g,1558628,1girl alternate_costume alternate_hairstyle ar...,iori_(blue_archive),blue_archive,https://cdn.donmai.us/360x360/83/a6/83a67e85d8...
3592,9145287,2025-04-13T07:00:29.491-04:00,61,g,530816,1girl animal_ears blush cat_choker cat_ears ch...,kazusa_(blue_archive),blue_archive,https://cdn.donmai.us/360x360/00/96/0096dd1137...
...,...,...,...,...,...,...,...,...,...
4659886,1349,2005-06-08T23:14:11.000-04:00,61,g,191489,cloud fireflies flower flower_focus full_moon ...,,original,https://cdn.donmai.us/360x360/dd/0b/dd0b253fd8...
4659938,1098,2005-06-06T15:46:38.000-04:00,64,g,112687,1girl blood blood_on_clothes blood_on_eyewear ...,claes,gunslinger_girl,https://cdn.donmai.us/360x360/64/e0/64e0b93183...
4659943,1090,2005-06-06T15:43:51.000-04:00,73,g,196925,2boys 2girls bag_charm black_footwear black_ha...,,original,https://cdn.donmai.us/360x360/11/14/1114351a7c...
4660162,145,2005-05-25T12:17:41.000-04:00,70,g,56957,2girls :d >_< apron bat_wings blonde_hair char...,kirisame_marisa remilia_scarlet,touhou,https://cdn.donmai.us/360x360/b6/68/b66899e64d...


In [17]:
# ---------------------------------
# Step 2: Build combined tags
# ---------------------------------

def combine_tags(row):
    tags = []

    for col in [
        "tag_string_general",
        "tag_string_character",
        "tag_string_copyright",
    ]:
        value = row[col]
        if pd.notna(value) and value:
            tags.extend(value.split())

    return tags

lite_df["tags"] = lite_df.apply(combine_tags, axis=1)

In [18]:
# ---------------------------------
# Step 3: Find top 50 tags
# ---------------------------------

counter = Counter()

for tags in lite_df["tags"]:
    counter.update(tags)

top50_tags = {tag for tag, _ in counter.most_common(50)}
top50_tags

{'1girl',
 '2girls',
 'animal_ears',
 'black_hair',
 'blonde_hair',
 'blue_archive',
 'blue_eyes',
 'blue_hair',
 'blush',
 'bow',
 'breasts',
 'brown_hair',
 'closed_eyes',
 'closed_mouth',
 'dress',
 'genshin_impact',
 'gloves',
 'grey_hair',
 'hair_between_eyes',
 'hair_ornament',
 'halo',
 'hat',
 'holding',
 'jacket',
 'jewelry',
 'long_hair',
 'long_sleeves',
 'looking_at_viewer',
 'multicolored_hair',
 'multiple_girls',
 'open_mouth',
 'pink_hair',
 'purple_eyes',
 'red_eyes',
 'ribbon',
 'school_uniform',
 'shirt',
 'short_hair',
 'sidelocks',
 'simple_background',
 'sitting',
 'skirt',
 'smile',
 'solo',
 'standing',
 'upper_body',
 'very_long_hair',
 'white_background',
 'white_hair',
 'white_shirt'}

In [19]:
# ---------------------------------
# Step 4: Keep images containing
# at least one top-50 tag
# ---------------------------------

lite_df = lite_df[
    lite_df["tags"].apply(
        lambda tags: any(tag in top50_tags for tag in tags)
    )
].copy()
# clean other tags
lite_df["tags"] = lite_df["tags"].apply(
    lambda tags: [tag for tag in tags if tag in top50_tags]
)
lite_df = lite_df[lite_df["tags"].str.len() > 0].reset_index(drop=True)
lite_df = lite_df[lite_df["tags"].str.len() >= 2].reset_index(drop=True)
lite_df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url,tags
0,9155913,2025-04-15T13:22:23.565-04:00,54,g,15591835,1girl black_sclera blonde_hair broken_mask col...,trigger_(zenless_zone_zero),zenless_zone_zero,https://cdn.donmai.us/360x360/33/9f/339f49cfdb...,"[1girl, blonde_hair, gloves, hair_ornament, op..."
1,9151749,2025-04-14T20:17:09.437-04:00,60,g,402201,1girl alternate_costume bare_shoulders black_h...,ellen_joe,zenless_zone_zero,https://cdn.donmai.us/360x360/63/86/6386a7c990...,"[1girl, black_hair, closed_mouth, hat, long_sl..."
2,9146829,2025-04-13T12:31:41.464-04:00,54,g,371585,1girl black_hair black_pantyhose blue_leotard ...,mona_(genshin_impact),genshin_impact,https://cdn.donmai.us/360x360/17/4c/174ca60b01...,"[1girl, black_hair, breasts, gloves, hat, jewe..."
3,9146730,2025-04-13T12:10:46.934-04:00,53,g,1558628,1girl alternate_costume alternate_hairstyle ar...,iori_(blue_archive),blue_archive,https://cdn.donmai.us/360x360/83/a6/83a67e85d8...,"[1girl, blush, closed_eyes, closed_mouth, grey..."
4,9145287,2025-04-13T07:00:29.491-04:00,61,g,530816,1girl animal_ears blush cat_choker cat_ears ch...,kazusa_(blue_archive),blue_archive,https://cdn.donmai.us/360x360/00/96/0096dd1137...,"[1girl, animal_ears, blush, hair_ornament, hal..."
...,...,...,...,...,...,...,...,...,...,...
42823,5634,2005-07-23T11:38:02.000-04:00,51,g,124002,1girl animification brown_eyes brown_hair clos...,hermione_granger,harry_potter_(series) wizarding_world,https://cdn.donmai.us/360x360/a3/54/a354f464ae...,"[1girl, brown_hair, closed_mouth, long_hair, l..."
42824,1098,2005-06-06T15:46:38.000-04:00,64,g,112687,1girl blood blood_on_clothes blood_on_eyewear ...,claes,gunslinger_girl,https://cdn.donmai.us/360x360/64/e0/64e0b93183...,"[1girl, blush, hair_ornament, holding, jacket,..."
42825,1090,2005-06-06T15:43:51.000-04:00,73,g,196925,2boys 2girls bag_charm black_footwear black_ha...,,original,https://cdn.donmai.us/360x360/11/14/1114351a7c...,"[2girls, black_hair, blush, brown_hair, closed..."
42826,145,2005-05-25T12:17:41.000-04:00,70,g,56957,2girls :d >_< apron bat_wings blonde_hair char...,kirisame_marisa remilia_scarlet,touhou,https://cdn.donmai.us/360x360/b6/68/b66899e64d...,"[2girls, blonde_hair, closed_eyes, dress, hat,..."


In [20]:
# ---------------------------------
# Step 5: Random sample
# ---------------------------------

lite_df = lite_df.sample(
    n=20_000,
    random_state=42,
).reset_index(drop=True)

# ---------------------------------
# Step 6: Train / Validation split
# ---------------------------------

train_df = lite_df.iloc[:16_000].reset_index(drop=True)
val_df = lite_df.iloc[16_000:].reset_index(drop=True)

print(len(train_df), len(val_df))

16000 4000


In [21]:
train_df.to_parquet(
    "danbooru2025_lite_train.parquet",
    index=False,
    compression="zstd",
)
val_df.to_parquet(
    "danbooru2025_lite_val.parquet",
    index=False,
    compression="zstd",
)